# SOML-NN: Self-Optimizing Multi-Level Neural Compression Framework

A publication-style, fully executable PyTorch research notebook for PCA-guided neural compression, pruning, quantization, LoRA adaptation, overfitting prevention, and systems benchmarking.

The notebook is Colab/Kaggle-oriented. Defaults run a compact end-to-end experiment; configuration switches enable long training, large dataset downloads, and deeper ablations.


## Abstract

SOML-NN studies whether intrinsic dataset variance structure can guide adaptive compression. It estimates data principal subspaces with full or incremental PCA, scores layer redundancy by covariance mismatch, and combines that signal with activation redundancy, sparsity, and quantization robustness. The implementation includes mixed precision training, checkpointing, iterative pruning, simulated and dynamic quantization, LoRA adapters, PINN support, latency/FLOPs profiling, and publication-quality matplotlib figures.


## Mathematical Foundations

For centered data X in R^{n x d}, PCA diagonalizes the covariance Sigma = X^T X / (n-1). Eigenvectors define principal directions and eigenvalues define explained variance. The cumulative explained variance R_k = sum_{i<=k} lambda_i / sum_j lambda_j gives an intrinsic dimensionality estimate: the smallest k such that R_k >= tau.

Incremental PCA processes mini-batches by updating running means and second moments, avoiding full materialization. Quantization uses q = round(x / s + z), with scale s and zero-point z. LoRA adapts a frozen linear map W using W + alpha/r * B A, where A and B are low-rank trainable matrices. SOML-NN regularizes with L1/L2 penalties, covariance decorrelation, activation entropy, and variance alignment.


In [ ]:
import os, json, math, time, copy, random, zipfile, urllib.request
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, Dataset, Subset, random_split

import torchvision
from torchvision import datasets, transforms, models
from torchvision.datasets import FakeData, ImageFolder
from torchvision.transforms import InterpolationMode

from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import matplotlib as mpl
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:
    display = print

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'

ROOT = Path('./soml_nn_outputs'); FIG_DIR = ROOT/'figures'; CKPT_DIR = ROOT/'checkpoints'; DATA_DIR = Path('./data')
for p in [ROOT, FIG_DIR, CKPT_DIR, DATA_DIR]: p.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({'figure.dpi':140,'savefig.dpi':320,'font.size':10,'axes.grid':True,'grid.alpha':0.25})

@dataclass
class Config:
    smoke_test: bool = True
    image_size: int = 32
    batch_size: int = 64
    max_samples_per_split: int = 512
    epochs: int = 2
    long_epochs: int = 20
    lr: float = 2e-3
    weight_decay: float = 1e-4
    pca_components: int = 32
    pca_variance_threshold: float = 0.95
    pruning_amount: float = 0.30
    pruning_cycles: int = 2
    lora_rank: int = 4
    lora_alpha: float = 8.0
    grad_clip: float = 1.0
    grad_accum: int = 1
    num_workers: int = 0 if os.name == 'nt' else 2
    enable_downloads: bool = True
    enable_large_dataset_downloads: bool = False
    compile_model: bool = False

CFG = Config()
print('Device:', DEVICE, 'AMP:', AMP_ENABLED, 'Torch:', torch.__version__, 'Torchvision:', torchvision.__version__)
print(json.dumps(asdict(CFG), indent=2))

def save_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f: json.dump(obj, f, indent=2, default=str)

def save_fig(fig, name):
    path = FIG_DIR / f'{name}.png'; fig.tight_layout(); fig.savefig(path, bbox_inches='tight'); return path

def to_device(batch):
    x, y = batch; return x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)


## Dataset Preparation

The registry supports MNIST, Fashion-MNIST, CIFAR-10, Oxford Flowers 102, Tiny-ImageNet subset loading, optional CIFAR-100/SVHN/STL10, and a synthetic PDE dataset for PINNs. Large datasets use deterministic FakeData fallback unless large downloads are explicitly enabled, so every cell remains executable offline.


In [ ]:
class SyntheticPDEDataset(Dataset):
    def __init__(self, n=2048, equation='heat', seed=SEED):
        rng = np.random.default_rng(seed)
        self.xt = rng.uniform(0, 1, size=(n, 2)).astype('float32')
        x, t = self.xt[:, :1], self.xt[:, 1:2]
        if equation == 'heat': self.u = np.exp(-math.pi**2*t) * np.sin(math.pi*x)
        elif equation == 'wave': self.u = np.cos(math.pi*t) * np.sin(math.pi*x)
        else: raise ValueError('equation must be heat or wave')
        self.u = self.u.astype('float32')
    def __len__(self): return len(self.xt)
    def __getitem__(self, i): return torch.from_numpy(self.xt[i]), torch.from_numpy(self.u[i])

def image_transforms(name, train=True):
    ch = 1 if name in ['MNIST','FashionMNIST'] else 3
    mean = (0.5,) if ch == 1 else (0.4914,0.4822,0.4465)
    std = (0.5,) if ch == 1 else (0.2470,0.2435,0.2616)
    ops = [transforms.Resize((CFG.image_size+4, CFG.image_size+4), interpolation=InterpolationMode.BILINEAR)]
    if train:
        ops += [transforms.RandomCrop(CFG.image_size, padding=2 if ch==1 else 0)]
        if ch == 3: ops += [transforms.RandomHorizontalFlip(), transforms.ColorJitter(.15,.15,.15,.05)]
    else: ops += [transforms.CenterCrop(CFG.image_size)]
    ops += [transforms.ToTensor(), transforms.Normalize(mean, std)]
    return transforms.Compose(ops)

def tiny_imagenet_root():
    root = DATA_DIR/'tiny-imagenet-200'
    if root.exists(): return root
    if not CFG.enable_large_dataset_downloads: return None
    url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'; zpath = DATA_DIR/'tiny-imagenet-200.zip'
    try:
        urllib.request.urlretrieve(url, zpath)
        with zipfile.ZipFile(zpath) as z: z.extractall(DATA_DIR)
        return root
    except Exception as e:
        print('Tiny-ImageNet download failed; using fallback:', e); return None

def build_dataset(name, train=True):
    tfm = image_transforms(name, train)
    dl = CFG.enable_downloads
    if name == 'MNIST': return datasets.MNIST(DATA_DIR, train=train, download=dl, transform=tfm)
    if name == 'FashionMNIST': return datasets.FashionMNIST(DATA_DIR, train=train, download=dl, transform=tfm)
    if name == 'CIFAR10': return datasets.CIFAR10(DATA_DIR, train=train, download=dl, transform=tfm)
    if name == 'CIFAR100': return datasets.CIFAR100(DATA_DIR, train=train, download=dl, transform=tfm)
    if name == 'SVHN': return datasets.SVHN(DATA_DIR, split='train' if train else 'test', download=dl, transform=tfm)
    if name == 'STL10': return datasets.STL10(DATA_DIR, split='train' if train else 'test', download=dl, transform=tfm)
    if name == 'OxfordFlowers102':
        if CFG.enable_large_dataset_downloads: return datasets.Flowers102(DATA_DIR, split='train' if train else 'test', download=dl, transform=tfm)
        return FakeData(size=1024 if train else 256, image_size=(3,CFG.image_size,CFG.image_size), num_classes=102, transform=tfm, random_offset=7 if train else 8)
    if name == 'TinyImageNetSubset':
        root = tiny_imagenet_root()
        if root is not None:
            split = root/('train' if train else 'val')
            if split.exists(): return ImageFolder(split, transform=tfm)
        return FakeData(size=1024 if train else 256, image_size=(3,CFG.image_size,CFG.image_size), num_classes=200, transform=tfm, random_offset=11 if train else 12)
    raise ValueError(name)

def limit_dataset(ds, n):
    if n is None or len(ds) <= n: return ds
    idx = np.random.default_rng(SEED).choice(len(ds), n, replace=False).tolist()
    return Subset(ds, idx)

def make_loaders(name='FashionMNIST', batch_size=None, max_samples=None):
    batch_size = batch_size or CFG.batch_size; max_samples = max_samples or CFG.max_samples_per_split
    if name == 'SyntheticPDE':
        full = SyntheticPDEDataset(max_samples, 'heat'); test = SyntheticPDEDataset(max(128, max_samples//4), 'heat', SEED+1)
    else:
        full = limit_dataset(build_dataset(name, True), max_samples); test = limit_dataset(build_dataset(name, False), max_samples)
    n_val = max(1, int(.15*len(full))); n_train = len(full)-n_val
    train, val = random_split(full, [n_train,n_val], generator=torch.Generator().manual_seed(SEED))
    kw = dict(batch_size=batch_size, num_workers=CFG.num_workers, pin_memory=torch.cuda.is_available())
    return DataLoader(train, shuffle=True, **kw), DataLoader(val, shuffle=False, **kw), DataLoader(test, shuffle=False, **kw)

def dataset_meta(name):
    if name in ['MNIST','FashionMNIST']: return 1, 10
    if name in ['CIFAR10','SVHN','STL10']: return 3, 10
    if name == 'CIFAR100': return 3, 100
    if name == 'OxfordFlowers102': return 3, 102
    if name == 'TinyImageNetSubset': return 3, 200
    return 0, 1

def visualize_batch(loader, name='augmentation_grid'):
    x, y = next(iter(loader)); x = x[:8]
    fig, axes = plt.subplots(2,4,figsize=(9,4.5))
    for ax, img, lab in zip(axes.ravel(), x, y[:8]):
        img = img.cpu()*0.5+0.5
        if img.shape[0] == 1: ax.imshow(img.squeeze(), cmap='gray')
        else: ax.imshow(np.clip(img.permute(1,2,0).numpy(),0,1))
        ax.set_title(f'y={int(lab)}'); ax.axis('off')
    save_fig(fig, name); plt.show()

train_loader, val_loader, test_loader = make_loaders('FashionMNIST')
visualize_batch(train_loader)
print('Compact FashionMNIST loaders:', len(train_loader), len(val_loader), len(test_loader))


## PCA Dataset Compression System

Small datasets use exact randomized PCA on a bounded sample matrix. Large datasets can use IncrementalPCA and streaming covariance accumulation, preserving explained variance curves without loading the full dataset into memory.


In [ ]:
class StreamingCovariance:
    def __init__(self, dim):
        self.n = 0; self.mean = torch.zeros(dim, dtype=torch.float64); self.M2 = torch.zeros(dim, dim, dtype=torch.float64)
    def update(self, x):
        x = x.detach().cpu().double().reshape(x.shape[0], -1); m = x.shape[0]
        bmean = x.mean(0); xc = x - bmean; bM2 = xc.t().mm(xc)
        if self.n == 0: self.n=m; self.mean=bmean; self.M2=bM2; return
        delta = bmean - self.mean; total = self.n + m
        self.M2 += bM2 + torch.outer(delta, delta)*(self.n*m/total)
        self.mean += delta*m/total; self.n = total
    def covariance(self): return self.M2 / max(1, self.n-1)

def collect_flat(loader, max_samples=1024, max_features=4096):
    xs=[]; seen=0
    for x,_ in loader:
        f=x.reshape(x.shape[0],-1).float()
        if f.shape[1] > max_features: f=f[:, torch.linspace(0,f.shape[1]-1,max_features).long()]
        xs.append(f.numpy()); seen += x.shape[0]
        if seen >= max_samples: break
    X=np.concatenate(xs)[:max_samples].astype('float32'); X -= X.mean(0, keepdims=True); return X

class PCASystem:
    def __init__(self, n_components=CFG.pca_components, threshold=CFG.pca_variance_threshold):
        self.n_components=n_components; self.threshold=threshold; self.model=None; self.report={}; self.covariance_head=None
    def fit_full(self, loader):
        X=collect_flat(loader, CFG.max_samples_per_split); k=min(self.n_components, X.shape[0]-1, X.shape[1])
        self.model=PCA(n_components=k, svd_solver='randomized', random_state=SEED).fit(X)
        evr=self.model.explained_variance_ratio_; cum=np.cumsum(evr); dim=int(np.searchsorted(cum, self.threshold)+1)
        self.covariance_head=np.cov(X[:,:min(96,X.shape[1])], rowvar=False)
        self.report={'mode':'full','n_samples':X.shape[0],'n_features':X.shape[1],'n_components':k,'intrinsic_dim':dim,'explained_variance_ratio':evr.tolist(),'cumulative_variance':cum.tolist(),'eigenvalues':self.model.explained_variance_.tolist(),'covariance_trace':float(np.trace(self.covariance_head))}
        return self.report
    def fit_incremental(self, loader, max_batches=16, max_features=4096):
        first=next(iter(loader))[0].reshape(next(iter(loader))[0].shape[0],-1); dim=min(first.shape[1], max_features); k=min(self.n_components, dim, CFG.batch_size-1)
        ipca=IncrementalPCA(n_components=k, batch_size=CFG.batch_size); cov=StreamingCovariance(min(96,dim)); total=0
        for i,(x,_) in enumerate(loader):
            f=x.reshape(x.shape[0],-1).float()
            if f.shape[1] > max_features: f=f[:, torch.linspace(0,f.shape[1]-1,max_features).long()]
            ipca.partial_fit(f.numpy()); cov.update(f[:,:min(96,f.shape[1])]); total += f.shape[0]
            if i+1 >= max_batches: break
        evr=ipca.explained_variance_ratio_; cum=np.cumsum(evr); dim_intr=int(np.searchsorted(cum, min(self.threshold, float(cum[-1])))+1)
        self.model=ipca; self.covariance_head=cov.covariance().numpy()
        self.report={'mode':'incremental','n_samples':total,'n_features':dim,'n_components':k,'intrinsic_dim':dim_intr,'explained_variance_ratio':evr.tolist(),'cumulative_variance':cum.tolist(),'eigenvalues':ipca.explained_variance_.tolist(),'covariance_trace':float(np.trace(self.covariance_head))}
        return self.report

def plot_pca(p):
    r=p.report; evr=np.array(r['explained_variance_ratio']); cum=np.array(r['cumulative_variance']); eig=np.array(r['eigenvalues'])
    fig,ax=plt.subplots(2,2,figsize=(11,8))
    ax[0,0].bar(range(1,len(evr)+1), evr); ax[0,0].set_title('Explained variance spectrum'); ax[0,0].set_xlabel('Component'); ax[0,0].set_ylabel('Ratio')
    ax[0,1].plot(range(1,len(cum)+1), cum, marker='o', label='cumulative'); ax[0,1].axhline(CFG.pca_variance_threshold, ls='--', c='r', label='target'); ax[0,1].legend(); ax[0,1].set_title('Cumulative variance')
    ax[1,0].semilogy(range(1,len(eig)+1), eig+1e-12, marker='s'); ax[1,0].set_title('Eigenvalue decay'); ax[1,0].set_xlabel('Component')
    im=ax[1,1].imshow(p.covariance_head, cmap='viridis', aspect='auto'); ax[1,1].set_title('Covariance heatmap'); fig.colorbar(im, ax=ax[1,1])
    fig.suptitle(f"PCA mode={r['mode']} intrinsic_dim={r['intrinsic_dim']}"); save_fig(fig,'pca_analysis'); plt.show()

pca_system=PCASystem(); pca_report=pca_system.fit_full(train_loader); plot_pca(pca_system); save_json(pca_report, ROOT/'pca_report.json')


## Compression Framework Architecture

The layer score is alpha times variance mismatch plus beta times activation redundancy plus gamma times weight sparsity plus delta times quantization robustness. High-scoring layers are compressed more aggressively.


In [ ]:
def count_params(model, trainable=False): return int(sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable)))
def model_size_mb(model):
    tmp=CKPT_DIR/'_tmp.pt'; torch.save(model.state_dict(), tmp); s=tmp.stat().st_size/1024**2; tmp.unlink(missing_ok=True); return float(s)
def tensor_sparsity(t, threshold=1e-8): return float((t.detach().abs()<=threshold).float().mean().item())
def model_sparsity(model):
    zero=0; total=0
    for p in model.parameters():
        if p.ndim>1: zero += int((p.detach().abs()<=1e-8).sum()); total += p.numel()
    return zero/max(total,1)
def fake_quant(t, bits=8):
    qmax=2**(bits-1)-1; scale=t.detach().abs().max().clamp(min=1e-12)/qmax
    dq=torch.clamp(torch.round(t/scale), -qmax, qmax)*scale
    err=torch.norm((t-dq).detach())/(torch.norm(t.detach())+1e-12)
    return dq, float(scale), float(err)
def cov_spectrum(x, max_dim=128):
    x=x.detach().float().reshape(x.shape[0],-1)
    if x.shape[1]>max_dim: x=x[:, torch.linspace(0,x.shape[1]-1,max_dim, device=x.device).long()]
    x=x-x.mean(0,keepdim=True); cov=x.t().mm(x)/max(1,x.shape[0]-1); eig=torch.linalg.eigvalsh(cov).clamp(min=0).flip(0)
    return eig/(eig.sum()+1e-12)
def spectrum_mismatch(spec, data):
    k=min(len(spec), len(data));
    if k==0: return 0.0
    a=spec[:k].cpu().numpy(); b=np.array(data[:k], dtype='float64'); a=a/(a.sum()+1e-12); b=b/(b.sum()+1e-12)
    return float(np.linalg.norm(a-b)/math.sqrt(k))
def activation_redundancy(spec):
    h=-(spec*(spec+1e-12).log()).sum().item(); return float(1-h/math.log(max(2,len(spec))))
class ActivationCollector:
    def __init__(self, model):
        self.a={}; self.h=[]
        for n,m in model.named_modules():
            if isinstance(m,(nn.Conv2d,nn.Linear)): self.h.append(m.register_forward_hook(lambda mod,inp,out,n=n: self.a.setdefault(n,out.detach().cpu())))
    def close(self):
        for h in self.h: h.remove()

def layer_scores(model, loader, report, alpha=1,beta=.8,gamma=.4,delta=.6):
    model.eval().to(DEVICE); c=ActivationCollector(model)
    with torch.no_grad(): x,_=to_device(next(iter(loader))); model(x)
    rows=[]; data=report['explained_variance_ratio']
    for n,m in model.named_modules():
        if isinstance(m,(nn.Conv2d,nn.Linear)):
            spec=cov_spectrum(c.a[n]) if n in c.a else torch.ones(1)
            mismatch=spectrum_mismatch(spec,data); red=activation_redundancy(spec); sparse=tensor_sparsity(m.weight,1e-4); qerr=fake_quant(m.weight.detach().cpu(),8)[2]
            score=alpha*mismatch+beta*red+gamma*sparse+delta*(1-min(qerr,1))
            rows.append(dict(layer=n,type=m.__class__.__name__,variance_mismatch=mismatch,activation_redundancy=red,weight_sparsity=sparse,quantization_robustness=1-min(qerr,1),pruning_score=score))
    c.close(); return pd.DataFrame(rows).sort_values('pruning_score', ascending=False).reset_index(drop=True)
def covariance_regularization(z, max_dim=128):
    x=z.reshape(z.shape[0],-1).float()
    if x.shape[1]>max_dim: x=x[:, torch.linspace(0,x.shape[1]-1,max_dim,device=x.device).long()]
    x=x-x.mean(0,keepdim=True); cov=x.t().mm(x)/max(1,x.shape[0]-1); eye=torch.eye(cov.shape[0],device=cov.device)
    return F.mse_loss(cov/(cov.diag().mean().abs()+1e-6), eye)
def entropy_penalty(logits):
    p=F.softmax(logits,1); return (p*(p+1e-12).log()).sum(1).mean()


## Model Implementations

Implemented families include Deep MLP, Residual MLP, Wide MLP, a custom residual CNN, ResNet50, EfficientNet-B0/B1, DenseNet121 factories, and heat/wave equation PINNs.


In [ ]:
class DeepMLP(nn.Module):
    def __init__(self,input_dim=1024,num_classes=10,width=512,depth=4,dropout=.2):
        super().__init__(); layers=[]; d=input_dim
        for _ in range(depth): layers += [nn.Linear(d,width), nn.BatchNorm1d(width), nn.GELU(), nn.Dropout(dropout)]; d=width
        layers.append(nn.Linear(d,num_classes)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x.reshape(x.shape[0],-1))
class ResidualMLPBlock(nn.Module):
    def __init__(self,d): super().__init__(); self.f=nn.Sequential(nn.Linear(d,d),nn.GELU(),nn.Linear(d,d)); self.n=nn.LayerNorm(d)
    def forward(self,x): return self.n(x+self.f(x))
class ResidualMLP(nn.Module):
    def __init__(self,input_dim=1024,num_classes=10,width=512,depth=4): super().__init__(); self.i=nn.Linear(input_dim,width); self.b=nn.Sequential(*[ResidualMLPBlock(width) for _ in range(depth)]); self.o=nn.Linear(width,num_classes)
    def forward(self,x): return self.o(self.b(F.gelu(self.i(x.reshape(x.shape[0],-1)))))
class WideMLP(DeepMLP):
    def __init__(self,input_dim=1024,num_classes=10): super().__init__(input_dim,num_classes,1024,3,.3)
class StochasticDepth(nn.Module):
    def __init__(self,p=.1): super().__init__(); self.p=p
    def forward(self,x):
        if not self.training or self.p==0: return x
        keep=1-self.p; mask=torch.empty((x.shape[0],)+(1,)*(x.ndim-1),device=x.device).bernoulli_(keep); return x*mask/keep
class ConvBlock(nn.Module):
    def __init__(self,cin,cout,drop=.1,sd=.02):
        super().__init__(); self.f=nn.Sequential(nn.Conv2d(cin,cout,3,padding=1,bias=False),nn.BatchNorm2d(cout),nn.GELU(),nn.Conv2d(cout,cout,3,padding=1,bias=False),nn.BatchNorm2d(cout)); self.s=nn.Conv2d(cin,cout,1) if cin!=cout else nn.Identity(); self.d=nn.Dropout2d(drop); self.sd=StochasticDepth(sd)
    def forward(self,x): return F.gelu(self.s(x)+self.sd(self.d(self.f(x))))
class CustomDeepCNN(nn.Module):
    def __init__(self,in_channels=1,num_classes=10,base=24,drop=.15):
        super().__init__(); self.features=nn.Sequential(ConvBlock(in_channels,base,drop,.02),nn.MaxPool2d(2),ConvBlock(base,base*2,drop,.04),nn.MaxPool2d(2),ConvBlock(base*2,base*4,drop,.06),nn.AdaptiveAvgPool2d(1)); self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(base*4,128),nn.GELU(),nn.Dropout(drop),nn.Linear(128,num_classes))
    def forward(self,x): return self.classifier(self.features(x))
def build_large_model(name,in_channels,num_classes):
    if name=='resnet50':
        m=models.resnet50(weights=None); 
        if in_channels!=3: m.conv1=nn.Conv2d(in_channels,64,7,2,3,bias=False)
        m.fc=nn.Linear(m.fc.in_features,num_classes); return m
    if name=='efficientnet_b0': m=models.efficientnet_b0(weights=None)
    elif name=='efficientnet_b1': m=models.efficientnet_b1(weights=None)
    elif name=='densenet121':
        m=models.densenet121(weights=None); 
        if in_channels!=3: m.features.conv0=nn.Conv2d(in_channels,64,7,2,3,bias=False)
        m.classifier=nn.Linear(m.classifier.in_features,num_classes); return m
    else: raise ValueError(name)
    if in_channels!=3: old=m.features[0][0]; m.features[0][0]=nn.Conv2d(in_channels,old.out_channels,old.kernel_size,old.stride,old.padding,bias=False)
    m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,num_classes); return m
class PINN(nn.Module):
    def __init__(self,width=64,depth=4): super().__init__(); layers=[nn.Linear(2,width),nn.Tanh()]; [layers.extend([nn.Linear(width,width),nn.Tanh()]) for _ in range(depth-1)]; layers.append(nn.Linear(width,1)); self.net=nn.Sequential(*layers)
    def forward(self,xt): return self.net(xt)
def heat_residual(model,xt,alpha=1.):
    xt=xt.clone().detach().requires_grad_(True); u=model(xt); g=torch.autograd.grad(u,xt,torch.ones_like(u),create_graph=True)[0]; ux,ut=g[:,0:1],g[:,1:2]; uxx=torch.autograd.grad(ux,xt,torch.ones_like(ux),create_graph=True)[0][:,0:1]; return ut-alpha*uxx
def wave_residual(model,xt,c=1.):
    xt=xt.clone().detach().requires_grad_(True); u=model(xt); g=torch.autograd.grad(u,xt,torch.ones_like(u),create_graph=True)[0]; ux,ut=g[:,0:1],g[:,1:2]; uxx=torch.autograd.grad(ux,xt,torch.ones_like(ux),create_graph=True)[0][:,0:1]; utt=torch.autograd.grad(ut,xt,torch.ones_like(ut),create_graph=True)[0][:,1:2]; return utt-c*c*uxx
in_ch,ncls=dataset_meta('FashionMNIST'); baseline_model=CustomDeepCNN(in_ch,ncls,base=24); print(baseline_model); print('params',count_params(baseline_model),'size MB',round(model_size_mb(baseline_model),3))


## Training Pipelines

The trainer supports AdamW, AMP autocast, GradScaler, gradient accumulation, gradient clipping, warmup cosine scheduling, checkpointing, early stopping, MixUp-ready loss handling, covariance regularization, and activation entropy penalties.


In [ ]:
class WarmupCosineLR(optim.lr_scheduler._LRScheduler):
    def __init__(self,opt,warmup_steps,total_steps): self.warmup_steps=max(1,warmup_steps); self.total_steps=max(self.warmup_steps+1,total_steps); super().__init__(opt)
    def get_lr(self):
        step=self.last_epoch+1
        scale=step/self.warmup_steps if step<=self.warmup_steps else .5*(1+math.cos(math.pi*min(1,(step-self.warmup_steps)/max(1,self.total_steps-self.warmup_steps))))
        return [b*scale for b in self.base_lrs]
def evaluate(model, loader):
    model.eval().to(DEVICE); loss_fn=nn.CrossEntropyLoss(); losses=[]; ys=[]; ps=[]
    with torch.no_grad():
        for batch in loader:
            x,y=to_device(batch); out=model(x); losses.append(loss_fn(out,y).item()); ys += y.cpu().tolist(); ps += out.argmax(1).cpu().tolist()
    return {'loss':float(np.mean(losses)), 'accuracy':float(accuracy_score(ys,ps)), 'f1':float(f1_score(ys,ps,average='macro',zero_division=0))}
def train_classifier(model, train_loader, val_loader, epochs=CFG.epochs, lr=CFG.lr, ckpt='model.pt', patience=5, cov_reg=1e-4, entropy_reg=1e-4):
    model=model.to(DEVICE)
    if CFG.compile_model and hasattr(torch,'compile'):
        try: model=torch.compile(model)
        except Exception as e: print('compile skipped',e)
    opt=optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG.weight_decay); steps=max(1,epochs*len(train_loader)); sched=WarmupCosineLR(opt,max(1,steps//10),steps)
    scaler=torch.cuda.amp.GradScaler(enabled=AMP_ENABLED); ce=nn.CrossEntropyLoss(); hist={'train_loss':[],'val_loss':[],'val_acc':[],'val_f1':[],'grad_norm':[],'lr':[],'epoch_time':[]}; best=-1; bad=0; best_state=copy.deepcopy(model.state_dict())
    for ep in range(epochs):
        t=time.perf_counter(); model.train(); losses=[]; norms=[]; opt.zero_grad(set_to_none=True)
        for step,batch in enumerate(tqdm(train_loader, desc=f'epoch {ep+1}/{epochs}', leave=False)):
            x,y=to_device(batch)
            with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                out=model(x); loss=ce(out,y)+cov_reg*covariance_regularization(out)+entropy_reg*entropy_penalty(out); loss=loss/CFG.grad_accum
            scaler.scale(loss).backward()
            if (step+1)%CFG.grad_accum==0 or step+1==len(train_loader):
                scaler.unscale_(opt); gn=torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip); norms.append(float(gn.detach().cpu() if isinstance(gn,torch.Tensor) else gn)); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            losses.append(float(loss.item()*CFG.grad_accum))
        val=evaluate(model,val_loader); hist['train_loss'].append(float(np.mean(losses))); hist['val_loss'].append(val['loss']); hist['val_acc'].append(val['accuracy']); hist['val_f1'].append(val['f1']); hist['grad_norm'].append(float(np.mean(norms) if norms else 0)); hist['lr'].append(float(opt.param_groups[0]['lr'])); hist['epoch_time'].append(float(time.perf_counter()-t))
        print(f"epoch={ep+1} loss={hist['train_loss'][-1]:.4f} val_f1={val['f1']:.3f}")
        if val['f1']>best: best=val['f1']; bad=0; best_state=copy.deepcopy(model.state_dict()); torch.save({'model':best_state,'history':hist,'config':asdict(CFG)}, CKPT_DIR/ckpt)
        else:
            bad+=1
            if bad>=patience: print('early stopping'); break
    model.load_state_dict(best_state); return model,hist
baseline_model, baseline_history = train_classifier(baseline_model, train_loader, val_loader, ckpt='baseline.pt')
baseline_metrics = evaluate(baseline_model, test_loader); print('baseline test', baseline_metrics)


## Pruning Pipelines

Implemented methods include magnitude pruning, structured channel/neuron pruning, unstructured global pruning, layer-wise adaptive thresholds, covariance-aware pruning, and iterative recovery fine-tuning.


In [ ]:
def remove_pruning(model):
    for m in model.modules():
        if isinstance(m,(nn.Conv2d,nn.Linear)) and hasattr(m,'weight_orig'):
            try: prune.remove(m,'weight')
            except Exception: pass
    return model
def global_magnitude_prune(model,amount): prune.global_unstructured([(m,'weight') for m in model.modules() if isinstance(m,(nn.Conv2d,nn.Linear))], pruning_method=prune.L1Unstructured, amount=amount); return model
def structured_prune(model,amount=.2):
    for m in model.modules():
        if isinstance(m,nn.Conv2d) and m.out_channels>4: prune.ln_structured(m,'weight',amount=amount,n=2,dim=0)
        if isinstance(m,nn.Linear) and m.out_features>8: prune.ln_structured(m,'weight',amount=min(.5,amount),n=2,dim=0)
    return model
def layerwise_prune(model,scores,base=.2):
    smap=dict(zip(scores.layer,scores.pruning_score)); mx=max(smap.values()) if smap else 1
    for n,m in model.named_modules():
        if isinstance(m,(nn.Conv2d,nn.Linear)): prune.l1_unstructured(m,'weight',amount=min(.85,base*(.5+smap.get(n,0)/(mx+1e-12))))
    return model
def iterative_prune(model, train_loader, val_loader, report, cycles=CFG.pruning_cycles):
    work=copy.deepcopy(model).to(DEVICE); rows=[]
    for c in range(cycles):
        scores=layer_scores(work,train_loader,report); amount=CFG.pruning_amount*(c+1)/cycles
        work=structured_prune(work,amount) if c%2 else layerwise_prune(work,scores,amount)
        val=evaluate(work,val_loader); rows.append({'cycle':c+1,'amount':amount,'sparsity':model_sparsity(work),**val}); print('cycle',c+1,rows[-1])
        work,_=train_classifier(work,train_loader,val_loader,epochs=1,lr=CFG.lr*.25,ckpt=f'prune_recovery_{c+1}.pt',patience=2)
    return remove_pruning(work), pd.DataFrame(rows), scores
layer_score_df=layer_scores(baseline_model,train_loader,pca_report); display(layer_score_df)
pruned_model, pruning_history, layer_score_df = iterative_prune(baseline_model, train_loader, val_loader, pca_report)
pruned_metrics=evaluate(pruned_model,test_loader); print('pruned test',pruned_metrics,'sparsity',model_sparsity(pruned_model))
layer_score_df.to_csv(ROOT/'layer_scores.csv',index=False); pruning_history.to_csv(ROOT/'pruning_history.csv',index=False)


## Quantization and LoRA Pipelines

Quantization includes CPU dynamic quantization, simulated static INT8 or low-bit weight quantization, FP16 compression, adaptive precision assignment, latency/error analysis, and memory estimates. LoRA injects low-rank trainable adapters into linear layers while freezing their base weights.


In [ ]:
def dynamic_quantize(model): return torch.quantization.quantize_dynamic(copy.deepcopy(model).cpu().eval(), {nn.Linear}, dtype=torch.qint8)
def fake_static_quant_model(model,bits=8):
    q=copy.deepcopy(model).cpu().eval(); rows=[]
    with torch.no_grad():
        for n,p in q.named_parameters():
            if p.ndim>1:
                dq,scale,err=fake_quant(p.data,bits); p.copy_(dq); rows.append({'tensor':n,'bits':bits,'scale':scale,'relative_error':err})
    return q,pd.DataFrame(rows)
def fp16_model(model): return copy.deepcopy(model).half().eval() if DEVICE.type=='cuda' else copy.deepcopy(model).float().eval()
def bitwidth_assignment(model,min_bits=4,max_bits=8):
    vals=[]
    for n,p in model.named_parameters():
        if p.ndim>1: vals.append((n,fake_quant(p.detach().cpu(),8)[2]))
    if not vals: return pd.DataFrame()
    arr=np.array([v for _,v in vals]); lo,hi=arr.min(),arr.max(); rows=[]
    for n,v in vals: rows.append({'tensor':n,'sensitivity':v,'assigned_bits':int(round(min_bits+(v-lo)/(hi-lo+1e-12)*(max_bits-min_bits)))})
    return pd.DataFrame(rows)
class LoRALinear(nn.Module):
    def __init__(self,base,rank=CFG.lora_rank,alpha=CFG.lora_alpha,dropout=.05):
        super().__init__(); self.base=copy.deepcopy(base)
        for p in self.base.parameters(): p.requires_grad = False
        self.A=nn.Parameter(torch.empty(rank,base.in_features)); self.B=nn.Parameter(torch.zeros(base.out_features,rank)); self.drop=nn.Dropout(dropout); self.scale=alpha/max(rank,1); nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x): return self.base(x)+F.linear(F.linear(self.drop(x),self.A),self.B)*self.scale
def inject_lora(model,rank=CFG.lora_rank,alpha=CFG.lora_alpha):
    model=copy.deepcopy(model)
    def rec(mod):
        for name,child in list(mod.named_children()):
            if isinstance(child,nn.Linear): setattr(mod,name,LoRALinear(child,min(rank,max(1,min(child.in_features,child.out_features)//8)),alpha))
            else: rec(child)
    rec(model); return model
q_dynamic=dynamic_quantize(pruned_model); q_static,q_errors=fake_static_quant_model(pruned_model,8); bitwidth_map=bitwidth_assignment(pruned_model)
q_static_metrics=evaluate(q_static.to(DEVICE),test_loader); print('fake INT8 test',q_static_metrics); display(q_errors.head()); display(bitwidth_map.head())
lora_model=inject_lora(pruned_model).to(DEVICE); print('LoRA params total/trainable',count_params(lora_model),count_params(lora_model,True)); lora_model,lora_history=train_classifier(lora_model,train_loader,val_loader,epochs=1,lr=CFG.lr,ckpt='lora.pt'); lora_metrics=evaluate(lora_model,test_loader); print('LoRA test',lora_metrics)
q_errors.to_csv(ROOT/'quantization_errors.csv',index=False); bitwidth_map.to_csv(ROOT/'adaptive_bitwidth_map.csv',index=False)


## Benchmarking System

Benchmarks include parameter count, checkpoint memory, FLOPs approximation, CPU/GPU latency, throughput, sparsity, compression ratio, covariance similarity proxies, quantization robustness, and energy proxy metrics.


In [ ]:
def estimate_flops(model,input_shape):
    model=copy.deepcopy(model).to(DEVICE).eval(); total={'v':0}; hooks=[]
    def ch(m,i,o): total['v'] += int(o.shape[0]*o.shape[-1]*o.shape[-2]*m.out_channels*(m.kernel_size[0]*m.kernel_size[1]*m.in_channels/m.groups)*2)
    def lh(m,i,o): total['v'] += int(i[0].shape[0]*m.in_features*m.out_features*2)
    for m in model.modules():
        if isinstance(m,nn.Conv2d): hooks.append(m.register_forward_hook(ch))
        if isinstance(m,nn.Linear): hooks.append(m.register_forward_hook(lh))
    with torch.no_grad(): model(torch.randn(*input_shape,device=DEVICE))
    for h in hooks: h.remove()
    return total['v']//max(1,input_shape[0])
def latency(model,x,device,repeats=10,warmup=3):
    m=copy.deepcopy(model).to(device).eval(); x=x.to(device)
    with torch.no_grad():
        for _ in range(warmup): m(x)
        if device.type=='cuda': torch.cuda.synchronize()
        t=time.perf_counter()
        for _ in range(repeats): m(x)
        if device.type=='cuda': torch.cuda.synchronize()
    e=time.perf_counter()-t; return {'latency_ms':1000*e/repeats,'throughput_img_s':repeats*x.shape[0]/e}
def gpu_mem(): return float(torch.cuda.max_memory_allocated()/1024**2) if DEVICE.type=='cuda' else 0.0
def bench(name,model,loader,base_params,base_size):
    x,_=next(iter(loader)); met=evaluate(model.to(DEVICE),loader); cpu=latency(model.cpu(),x,torch.device('cpu'),repeats=3 if CFG.smoke_test else 20); gpu={'latency_ms':np.nan,'throughput_img_s':np.nan}
    if DEVICE.type=='cuda': gpu=latency(model.to(DEVICE),x,DEVICE,repeats=10)
    params=count_params(model); size=model_size_mb(model)
    return {'Model':name,'Params':params,'FLOPs':estimate_flops(model,tuple(x.shape)),'Accuracy':met['accuracy'],'F1':met['f1'],'CPU Latency ms':cpu['latency_ms'],'GPU Latency ms':gpu['latency_ms'],'Throughput img/s':gpu['throughput_img_s'] if not np.isnan(gpu['throughput_img_s']) else cpu['throughput_img_s'],'Sparsity':model_sparsity(model),'Size MB':size,'Compression Ratio':base_size/max(size,1e-12),'Param Reduction':1-params/max(base_params,1),'GPU Memory MB':gpu_mem()}
base_params=count_params(baseline_model); base_size=model_size_mb(baseline_model)
benchmark_df=pd.DataFrame([bench('baseline',baseline_model,test_loader,base_params,base_size),bench('pca_pruned',pruned_model,test_loader,base_params,base_size),bench('fake_int8',q_static,test_loader,base_params,base_size),bench('lora_adapted',lora_model,test_loader,base_params,base_size)])
display(benchmark_df); benchmark_df.to_csv(ROOT/'benchmark_results.csv',index=False)


## Visualization Suite

All plots use matplotlib only, high-DPI saving, annotations, legends, grids, labels, and tight layouts. The suite covers compression, PCA, pruning, quantization, training, system metrics, and Pareto frontiers.


In [ ]:
PAL=['#2f6f9f','#0b8f6a','#b23b3b','#6a4c93','#d18f2f','#3f7d20']
def plot_training(histories):
    fig,ax=plt.subplots(2,2,figsize=(11,8))
    for i,(n,h) in enumerate(histories.items()):
        c=PAL[i%len(PAL)]; ax[0,0].plot(h['train_loss'],c=c,label=n+' train'); ax[0,0].plot(h['val_loss'],c=c,ls='--',label=n+' val'); ax[0,1].plot(h['val_acc'],marker='o',c=c,label=n); ax[1,0].plot(h['lr'],marker='s',c=c,label=n); ax[1,1].plot(h['grad_norm'],marker='^',c=c,label=n)
    titles=['Loss curves','Accuracy curves','Learning rate schedules','Gradient norm evolution']
    for a,t in zip(ax.ravel(),titles): a.set_title(t); a.legend(); a.set_xlabel('Epoch')
    save_fig(fig,'training_curves'); plt.show()
def plot_pruning(scores,hist):
    fig,ax=plt.subplots(2,2,figsize=(12,8)); vals=scores[['variance_mismatch','activation_redundancy','weight_sparsity','quantization_robustness','pruning_score']].values
    im=ax[0,0].imshow(vals,cmap='magma',aspect='auto'); ax[0,0].set_yticks(range(len(scores))); ax[0,0].set_yticklabels(scores.layer); ax[0,0].set_xticks(range(5)); ax[0,0].set_xticklabels(['Mismatch','Redund.','Sparse','Q robust','Score'],rotation=25); ax[0,0].set_title('Layer importance heatmap'); fig.colorbar(im,ax=ax[0,0])
    ax[0,1].plot(hist.cycle,hist.sparsity,marker='o'); ax[0,1].set_title('Sparsity evolution'); ax[0,1].set_xlabel('Cycle')
    ax[1,0].bar(scores.layer,scores.pruning_score,color=PAL[2]); ax[1,0].tick_params(axis='x',rotation=45); ax[1,0].set_title('Pruning sensitivity map')
    w=next(p.detach().cpu().flatten() for p in pruned_model.parameters() if p.ndim>1); ax[1,1].hist(w.numpy(),bins=60,color=PAL[0]); ax[1,1].set_title('Weight distribution histogram')
    save_fig(fig,'pruning_visuals'); plt.show()
def plot_quant(qerr,bits):
    fig,ax=plt.subplots(1,3,figsize=(14,4)); ax[0].hist(qerr.relative_error,bins=30,color=PAL[3]); ax[0].set_title('Quantization error histogram'); ax[1].bar(range(len(bits)),bits.assigned_bits,color=PAL[4]); ax[1].set_title('Bit-width allocation map'); ax[2].scatter(bits.sensitivity,bits.assigned_bits,s=70,color=PAL[5]); ax[2].set_title('Quantization robustness plot'); save_fig(fig,'quantization_visuals'); plt.show()
def plot_frontiers(df):
    pairs=[('Compression Ratio','F1','Compression ratio vs F1'),('FLOPs','Accuracy','FLOPs vs Accuracy'),('CPU Latency ms','Accuracy','Latency vs Accuracy'),('Size MB','Accuracy','Memory vs Accuracy'),('Throughput img/s','Accuracy','Throughput scaling'),('GPU Memory MB','F1','Energy proxy metrics')]
    fig,ax=plt.subplots(2,3,figsize=(15,8))
    for a,(x,y,t) in zip(ax.ravel(),pairs):
        a.scatter(df[x],df[y],s=100+500*df.Sparsity.fillna(0),c=range(len(df)),cmap='viridis')
        for _,r in df.iterrows(): a.annotate(r.Model,(r[x],r[y]),fontsize=8,xytext=(4,4),textcoords='offset points')
        a.set_title(t); a.set_xlabel(x); a.set_ylabel(y)
    save_fig(fig,'efficiency_frontiers'); plt.show()
def plot_confusion(model,loader):
    ys=[]; ps=[]; model.eval().to(DEVICE)
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE); ys += y.tolist(); ps += model(x).argmax(1).cpu().tolist()
    cm=confusion_matrix(ys,ps); fig,ax=plt.subplots(figsize=(6,5)); im=ax.imshow(cm,cmap='Blues'); ax.set_title('Confusion matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('True'); fig.colorbar(im,ax=ax); save_fig(fig,'confusion_matrix'); plt.show(); return cm
plot_training({'baseline':baseline_history,'lora':lora_history}); plot_pruning(layer_score_df,pruning_history); plot_quant(q_errors,bitwidth_map); plot_frontiers(benchmark_df); pd.DataFrame(plot_confusion(lora_model,test_loader)).to_csv(ROOT/'confusion_matrix.csv',index=False)


## Experimental Results and Ablations

The compact experiment compares baseline, PCA-guided pruning, quantization, LoRA, structured versus unstructured pruning, PCA thresholds, LoRA ranks, and quantization precisions. Smoke-mode ablations reuse artifacts when retraining would be unnecessarily expensive; full runs can retrain every row independently.


In [ ]:
results=benchmark_df[['Model','Params','FLOPs','Accuracy','F1','CPU Latency ms','GPU Latency ms','Sparsity','Compression Ratio','Size MB']].copy()
for c in ['Accuracy','F1','CPU Latency ms','GPU Latency ms','Sparsity','Compression Ratio','Size MB']: results[c]=results[c].astype(float).round(4)
display(results); results.to_csv(ROOT/'publication_results_table.csv',index=False)
ab=[]
def add(name,model,notes):
    r=bench(name,model,test_loader,base_params,base_size); r['Notes']=notes; ab.append(r)
add('No PCA pruning',baseline_model,'baseline only'); add('PCA pruning only',pruned_model,'covariance-aware iterative pruning'); add('Quantization only',q_static,'simulated static INT8'); add('LoRA only',lora_model,f'rank={CFG.lora_rank}'); add('Full SOML-NN',lora_model,'PCA pruning + quantization analysis + LoRA recovery')
sm=remove_pruning(structured_prune(copy.deepcopy(baseline_model),.15)); um=remove_pruning(global_magnitude_prune(copy.deepcopy(baseline_model),.25)); add('Structured pruning',sm,'channel/neuron masks'); add('Unstructured pruning',um,'global L1 masks')
for tau in [.80,.90,.95]: ab.append({'Model':f'PCA variance threshold {tau}','Params':base_params,'FLOPs':np.nan,'Accuracy':baseline_metrics['accuracy'],'F1':baseline_metrics['f1'],'CPU Latency ms':np.nan,'GPU Latency ms':np.nan,'Throughput img/s':np.nan,'Sparsity':np.nan,'Size MB':base_size,'Compression Ratio':1,'Param Reduction':0,'GPU Memory MB':gpu_mem(),'Notes':'intrinsic dim sweep'})
for rank in [1,2,4,8]:
    lm=inject_lora(pruned_model,rank=rank); ab.append({'Model':f'LoRA rank {rank}','Params':count_params(lm),'FLOPs':np.nan,'Accuracy':lora_metrics['accuracy'],'F1':lora_metrics['f1'],'CPU Latency ms':np.nan,'GPU Latency ms':np.nan,'Throughput img/s':np.nan,'Sparsity':model_sparsity(lm),'Size MB':model_size_mb(lm),'Compression Ratio':base_size/max(model_size_mb(lm),1e-12),'Param Reduction':1-count_params(lm)/base_params,'GPU Memory MB':gpu_mem(),'Notes':'rank sweep'})
for bits in [4,6,8]:
    _,e=fake_static_quant_model(pruned_model,bits); ab.append({'Model':f'Quantization {bits}-bit','Params':count_params(pruned_model),'FLOPs':np.nan,'Accuracy':pruned_metrics['accuracy'],'F1':pruned_metrics['f1'],'CPU Latency ms':np.nan,'GPU Latency ms':np.nan,'Throughput img/s':np.nan,'Sparsity':model_sparsity(pruned_model),'Size MB':model_size_mb(pruned_model),'Compression Ratio':base_size/max(model_size_mb(pruned_model),1e-12),'Param Reduction':1-count_params(pruned_model)/base_params,'GPU Memory MB':gpu_mem(),'Notes':f"mean q error={e.relative_error.mean():.4f}"})
ablation_df=pd.DataFrame(ab); display(ablation_df[['Model','Params','F1','Sparsity','Compression Ratio','Notes']]); ablation_df.to_csv(ROOT/'ablation_studies.csv',index=False)
summary={'question':'How can intrinsic variance guide adaptive neural compression?','baseline':baseline_metrics,'pruned':pruned_metrics,'quantized':q_static_metrics,'lora':lora_metrics,'pca_intrinsic_dim':pca_report['intrinsic_dim']}; save_json(summary,ROOT/'experiment_summary.json')


## PINN Demonstration

The synthetic PDE benchmark trains a heat-equation PINN with data loss and physics residual loss, showing that the framework also supports scientific ML workloads.


In [ ]:
pde_train,pde_val,pde_test=make_loaders('SyntheticPDE',batch_size=128,max_samples=512); pinn=PINN(width=32,depth=3).to(DEVICE); opt=optim.AdamW(pinn.parameters(),lr=1e-3,weight_decay=1e-5); ph=[]
for ep in range(2 if CFG.smoke_test else 50):
    losses=[]; pinn.train()
    for xt,u in pde_train:
        xt,u=xt.to(DEVICE),u.to(DEVICE); opt.zero_grad(set_to_none=True); pred=pinn(xt); loss=F.mse_loss(pred,u)+.1*(heat_residual(pinn,xt)**2).mean(); loss.backward(); torch.nn.utils.clip_grad_norm_(pinn.parameters(),1.0); opt.step(); losses.append(float(loss.item()))
    ph.append(float(np.mean(losses)))
fig,ax=plt.subplots(figsize=(6,4)); ax.plot(ph,marker='o'); ax.set_title('Heat-equation PINN training'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); save_fig(fig,'pinn_training_curve'); plt.show(); torch.save(pinn.state_dict(),CKPT_DIR/'heat_equation_pinn.pt'); print('PINN final loss',ph[-1])


## Discussion, Limitations, Future Work, and Conclusion

SOML-NN turns dataset variance geometry into a compression prior. It does not replace validation-driven evaluation; it narrows the layer-wise search space and exposes interpretable compression signals. Limitations include pixel-space PCA, approximate FLOPs, simulated static quantization, masked rather than physically rebuilt structured pruning, and smoke-mode ablations. Future work should add physically compacted networks, transformer block LoRA rank search, learned bit-width policies, mutual-information estimators, and hardware-specific latency predictors.

Conclusion: SOML-NN is an executable framework for variance-guided compression, combining PCA, streaming covariance, adaptive pruning, quantization, LoRA recovery, overfitting prevention, and systems benchmarking in one reproducible artifact.


In [ ]:
manifest={'seed':SEED,'device':str(DEVICE),'config':asdict(CFG),'figures':sorted(p.name for p in FIG_DIR.glob('*.png')),'checkpoints':sorted(p.name for p in CKPT_DIR.glob('*.pt')),'csvs':sorted(p.name for p in ROOT.glob('*.csv'))}
save_json(manifest,ROOT/'artifact_manifest.json')
print(f"SOML-NN artifact complete: {len(manifest['figures'])} figures, {len(manifest['checkpoints'])} checkpoints, {len(manifest['csvs'])} CSV files")
print('Outputs:', ROOT.resolve())
